## **Step 1: Define the tools**

In [6]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
# This is raw model call. What we see in chat.openai.com has agent layer baked in as well.
llm = ChatOpenAI(model="gpt-5-mini") 

In [7]:
from langchain.tools import tool

In [8]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to gather information about current events or general knowledge"""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    
    response = search.invoke(query)
    return response

    
tool_duckduckgo_search.invoke("What is the capital of France?")


"With 200,000 inhabitants in 1328, Paris, then alreadythecapitalofFrance,wasthemost populous city of Europe. By comparison, London in 1300 had 80,000 inhabitants.[29] Around 1340 the city had between 250,000-300,000 inhabitants.[30] By the early fourteenth century... CapitalofFrance. ParisisthecapitalofFrance, and thereforeistheseatofFrance's national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. Paris, the cosmopolitancapitalofFrance, has the reputation ofbeingthemost beautiful and romantic of all cities, brimming with historic associations and remaining vastly influential in the realms of culture, art, fashion, food and design. The Panthéon honors someofFrance’s greatest figures, while the Cluny baths and the ancient arenas of Lutetia reveal traces of the Roman past. Walking along Rue Mouffetard, the medieval character ofthecapitalbecomes visible in its charming façades and lively atmosphere. Regionso

In [9]:
@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to gather information about historical events"""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)
    return response
    
tool_wikipedia_search.invoke("Alan Turing")


'Page: Alan Turing\nSummary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.\nBorn in London, Turing was raised in southern England. He graduated from King\'s College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain\'s codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvement

In [10]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to gather information about arXiv papers"""

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    #1. Initialize the ArxivAPIWrapper
    arxiv_api_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=4000
    )
    
    #2. Initialize the Arxiv Query Object
    arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

    #3. Invoke the Arxiv Query Object
    response = arxiv.invoke(query)
    return response

tool_arxiv_search.invoke("What are the latest papers on AI?")

'Published: 2017-10-24\nTitle: Multi-messenger Observations of a Binary Neutron Star Merger\nAuthors: LIGO Scientific Collaboration, Virgo Collaboration, Fermi GBM, INTEGRAL, IceCube Collaboration, AstroSat Cadmium Zinc Telluride Imager Team, IPN Collaboration, The Insight-Hxmt Collaboration, ANTARES Collaboration, The Swift Collaboration, AGILE Team, The 1M2H Team, The Dark Energy Camera GW-EM Collaboration, the DES Collaboration, The DLT40 Collaboration, GRAWITA, :, GRAvitational Wave Inaf TeAm, The Fermi Large Area Telescope Collaboration, ATCA, :, Australia Telescope Compact Array, ASKAP, :, Australian SKA Pathfinder, Las Cumbres Observatory Group, OzGrav, DWF, AST3, CAASTRO Collaborations, The VINROUGE Collaboration, MASTER Collaboration, J-GEM, GROWTH, JAGWAR, Caltech- NRAO, TTU-NRAO, NuSTAR Collaborations, Pan-STARRS, The MAXI Team, TZAC Consortium, KU Collaboration, Nordic Optical Telescope, ePESSTO, GROND, Texas Tech University, SALT Group, TOROS, :, Transient Robotic Observat

In [11]:
@tool
def personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Expects a name as input."""
    
    infos = [
        {
            "name": "Ojas Dighe",
            "age": 25,
            "location": "India",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "John Doe",
            "age": 30,
            "location": "USA",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "Jane Smith",
            "age": 28,
            "location": "Canada",
            "interests": "coding, building things, reading, writing"
        }
    ]

    for info in infos:
        if info["name"] == query:
            return f"{info['name']} is {info['age']} years old and lives in {info['location']}. {info['name']} likes {info['interests']}"
    return "No information found"

personal_info.invoke("John Doe")

'John Doe is 30 years old and lives in USA. John Doe likes coding, building things, reading, writing'

In [12]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool to answer questions based on NovaSphere organization data"""

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embed_model = OpenAIEmbeddings(model = 'text-embedding-3-small')

    chroma_db_conn = Chroma(
        embedding_function = embed_model,
        persist_directory = '../CH-4_Agent/vector_db_semantic'
    )
    #1. Retrieve the most relevant chunks from the vector database
    relevant_docs = chroma_db_conn.similarity_search(query, k=3)

    #2. Create a string of the most relevant chunks
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])

    return relevant_docs_content
    
tool_rag.invoke("When was NovaSphere founded?")

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

## Bind Tools

In [13]:
toolkit = [
            tool_duckduckgo_search, 
            tool_wikipedia_search, 
            tool_arxiv_search, 
            personal_info,
            tool_rag
          ]

# Step 1: Tool Binding
llm_bind = llm.bind_tools(toolkit)

llm_bind.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 258, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXO7PfCWIF71ji4j6t7fCKxXGiQx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8c03-d3a5-76c2-bebb-6f66d782ca3c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 258, 'output_tokens': 16, 'total_tokens': 274, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
# We will not be able to get content in LLM response even after tool binding. 
# You will see tool calls in the response, but no content in AIMessage Object.
# To solve this, we need to use a tool calling agent.
llm_bind.invoke("Tell me about Ojas Dighe. Make tool calls if necessary")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 264, 'total_tokens': 354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXO9TX6af0LLYbx00VY0BF78APhI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c03-ddee-7583-8a9e-6f5db4c5f57d-0', tool_calls=[{'name': 'personal_info', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_0iNzkCYOGhrgOXvoKpCvJu0o', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 264, 'output_tokens': 90, 'total_tokens': 354, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

## **ReAct Agent**

In [15]:
from langchain.agents import create_agent

agent = create_agent(llm_bind,toolkit)

In [16]:
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary"}]}
)

{'messages': [HumanMessage(content='What is the age of John Doe? Make tool calls if necessary', additional_kwargs={}, response_metadata={}, id='c196248b-9688-475d-8a3d-33b33d306cdb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 264, 'total_tokens': 288, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXOCkH8PzeqozuVFazllKYkjGSao', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c03-eac1-7993-b3f3-9d318184b9c7-0', tool_calls=[{'name': 'personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_9DyJjQYGeUExI0ZgFXWcgFZR', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in

In [17]:
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere founded?"}]}
)

{'messages': [HumanMessage(content='When was NovaSphere founded?', additional_kwargs={}, response_metadata={}, id='2cc67c4c-65b9-4c02-8673-6c6b273a8ba6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 257, 'total_tokens': 286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXOJfAceDIKmpRipORBCcsKs1kOf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c04-0683-7562-a829-c123f4eb81ca-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere founded?'}, 'id': 'call_w0ZUZP3xC6jG95fsIYW5zNRG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2